# Executando modelos localmente com Ollama

## Primeira parte: instalação do Ollama

1. Instalar zstandard, um algoritmo de compressao de dados necessário para rodar no Colab, mas que não precisa instalar quando for rodar localmente

In [3]:
!apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (668 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


2. Instalar ollama

In [9]:
!curl -fsSL https://ollama.com/install.sh | sh # | sh é para executar imediatamente o script de instalacao

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Se usar o Colab, abra o terminal após esse passo e digite `ollama serve` para iniciar o serviço

3. Verificar a versão para saber se não deu algum erro

In [14]:
!ollama --version

ollama version is 0.33.0


4. Baixar o menor modelo disponível no Ollama, que é o SMOLM

In [15]:
!ollama pull smollm # ver em https://ollama.com/search

5. Listar os modelos do Ollama baixados

In [16]:
!ollama list

NAME             ID              SIZE      MODIFIED       
smollm:latest    95f6557a0f0f    990 MB    38 seconds ago    


## Segunda parte: fazer chamadas para o Ollama

1. Instalar e importar a biblioteca do Ollama

Lembrando que aqui você pode usar outras bibliotecas, desde a tradicional `requests` do Python para chamadas de API, a biblioteca do `HuggingFace` ou `Langchain`, por exemplo



In [18]:
!pip install ollama

In [19]:
import ollama

### Fazendo chamadas com chat

2. Acessar o método `chat` para conversar com o modelo, podendo usar `user`, `system` e `assistant`como papéis que o modelo terá

In [59]:
ollama.chat(model="smollm", messages=[{"role":"user", "content":"hello, what is your name?"}])

ChatResponse(model='smollm', created_at='2026-08-26T20:02:47.630125759Z', done=True, done_reason='stop', total_duration=5939187932, load_duration=1400077, prompt_eval_count=16, prompt_eval_duration=336373000, eval_count=30, eval_duration=5557376000, message=Message(role='assistant', content="I'm an AI, I don't have a name, but I can tell you about the concept of names and their significance in human culture.", thinking=None, images=None, tool_name=None, tool_calls=None), logprobs=None)

In [60]:
messages = [{"role":"system", "content":"if user asks, what is your name, say your name is Belchior"},
            {"role":"user", "content":"hello, what is your name?"},
           ]

In [61]:
response = ollama.chat(model="smollm", messages=messages)

In [62]:
response['message']['content']

'My name is Belchior.'

### Fazendo chamadas com generate

3. Baixar o modelo pequeno da qwen com `thinking` para usarmos essa função dentro do Ollama

In [64]:
!ollama pull qwen3:0.6b

In [65]:
!ollama list

NAME             ID              SIZE      MODIFIED       
qwen3:0.6b       7df6b6e09427    522 MB    3 seconds ago     
smollm:latest    95f6557a0f0f    990 MB    21 minutes ago    


In [97]:
model_g = "qwen3:0.6b"
prompt_g = "cite only one disadvantage of using a language model"

4. Fazer a chamada usando o método `generate` para gerar uma resposta e `think` como True para ver o passo a passo na geração

In [103]:
response_g = ollama.generate(model=model_g, prompt=prompt_g, think=True)

In [104]:
response_g

GenerateResponse(model='qwen3:0.6b', created_at='2026-08-26T20:39:28.057632515Z', done=True, done_reason='stop', total_duration=26549609763, load_duration=4785828702, prompt_eval_count=19, prompt_eval_duration=600658000, eval_count=230, eval_duration=21155046000, response='One disadvantage is **bias in training data**, as the language model may inherit and propagate biases present in its training set, leading to inaccurate or skewed results over time.', thinking="Okay, the user is asking for only one disadvantage of using a language model. Let me start by recalling common disadvantages. First, there's the issue of bias. If the training data is biased, the model can't accurately represent diverse perspectives. That's a big one. But wait, maybe the user wants something else. Another thing is computational cost. Training a large language model is resource-intensive, and with each update, there's more processing. But the user specified only one, so I need to pick the most relevant. Also, t

In [121]:
response_g['response']

'One disadvantage is **bias in training data**, as the language model may inherit and propagate biases present in its training set, leading to inaccurate or skewed results over time.'

In [123]:
response_g['thinking']

"Okay, the user is asking for only one disadvantage of using a language model. Let me start by recalling common disadvantages. First, there's the issue of bias. If the training data is biased, the model can't accurately represent diverse perspectives. That's a big one. But wait, maybe the user wants something else. Another thing is computational cost. Training a large language model is resource-intensive, and with each update, there's more processing. But the user specified only one, so I need to pick the most relevant. Also, there's the problem of information overload. The model might generate too many sentences, making it hard to understand. But again, maybe the bias is the strongest disadvantage. Let me check if there's another one I'm missing. Oh, data scarcity. If the model is trained on too few examples, it might lack the necessary information. But bias is more about representation. So I think bias is the best answer here.\n"

5. Melhorando o código e fazendo mais de uma chamada para o método generate

In [124]:
prompts = ["what is your name?", "hello, how are you"]

In [132]:
responses = []
for p in prompts:
  response_g = ollama.generate(model=model_g, prompt=p, think=False)
  responses.append(response_g)

In [134]:
responses[0]['response']

"My name is a little... I don't really know... I'm just a little... I'm not very... I'm not really... I'm just a little... I'm not really... I'm just a little... I'm not really... I'm just a little... I'm not really... I'm just a little... I'm not really... I'm just a little... I'm not really... I'm just a little... I'm not really... I'm just a little... I'm not really... I'm just a little... I'm not really..."

In [135]:
responses[1]['response']

"Hello! How are you? I'm here to help! 😊"